# Context Managers - Stop Leaking Resources

## The Problem Context Managers Solve
You open a database connection. Your code crashes. The connection never closes. Now you have a resource leak.

You open 10,000 files in a loop. Forget to close one. Eventually you hit the OS file descriptor limit. Your program crashes.

__This is amateur hour.__

__Professional solution:__ Context managers automatically clean up resources, even when errors occur.

## Part 1: The Problem - Manual Resource Management Sucks

### Back Code (Don't Do This)

In [ ]:
# Open file
f = open('data.csv', 'r')
data = f.read()
# ... process data ...
f.close()  # What if an error happens before this? FILE LEAK.

# Database connection
conn = db.connect('postgresql://...')
cursor = conn.cursor()
cursor.execute("SELECT * FROM users")
# ... process results ...
cursor.close()
conn.close()  # What if query crashes? CONNECTION LEAK.

__Problems:__
1. If error happens, cleanup never runs
2. Easy to forget to cleanup code
3. Code is cluttered with open/close logic
4. Not exception-safe

## Part 2: The with Statement (Context Managers)

### Good Code (Do This)

In [ ]:
# File automatically closes, even on error
with open('data.csv', 'r') as f:
    data = f.read()
    # Process data
    # If error occurs here, file STILL closes

# Database connection automatically closes
with db.connect('postgresql://...') as conn:
    with conn.cursor() as cursor:
        cursor.execute("SELECT * FROM users")
        # Process results
    # Cursor closes here
# Connection closes here, ALWAYS

__Benefits:__
- Resource ALWAYS cleaned up
- Exception-safe by default
- Cleaner, more readable code
- Standard Python pattern

## Part 3: How Context Managers Work
A context manager is any object with _ _ enter _ _() and _ _ exit _ _() methods.

In [ ]:
class MyContextManager:
    def __enter__(self):
        """Called when entering 'with' block."""
        print("Setting up resource")
        return self  # This is what 'as var' gets
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """Called when exiting 'with' block."""
        print("Cleaning up resource")
        # exc_type: Exception class if error occurred, else None
        # exc_val: Exception instance if error occurred, else None
        # exc_tb: Traceback if error occurred, else None
        return False  # False = propagate exceptions, True = suppress

# Usage
with MyContextManager() as cm:
    print("Inside with block")
    # Automatic cleanup happens after this

__Execution flow:__
1. _ _ enter _ _() runs-setup
2. Code inside with block runs
3. _ _ exit _ _() runs-cleanup(ALWAYS, even on error)

## Part 4: Real Example - Database Connection

In [ ]:
import psycopg2

class DatabaseConnection:
    """Context manager for database connections."""
    
    def __init__(self, connection_string: str):
        self.connection_string = connection_string
        self.conn = None
        self.cursor = None
    
    def __enter__(self):
        """Open connection."""
        print("Opening database connection")
        self.conn = psycopg2.connect(self.connection_string)
        self.cursor = self.conn.cursor()
        return self.cursor  # Return cursor for queries
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """Close connection, commit or rollback."""
        if exc_type is not None:
            # Error occurred - rollback
            print(f"Error occurred: {exc_val}")
            self.conn.rollback()
        else:
            # Success - commit
            self.conn.commit()
        
        # Always close
        if self.cursor:
            self.cursor.close()
        if self.conn:
            self.conn.close()
        print("Database connection closed")
        
        return False  # Don't suppress exceptions

# Usage
with DatabaseConnection('postgresql://localhost/mydb') as cursor:
    cursor.execute("INSERT INTO users (name) VALUES ('Alice')")
    cursor.execute("INSERT INTO users (name) VALUES ('Bob')")
    # If error here, automatic rollback
    # If success, automatic commit
# Connection automatically closed

__key insight:__ Transaction handling (commit/rollback) happens automatically based on whether an error occurred.

## Part 5: Creating Context Managers with @contextmanager
Writing classes is verbose. Use @contextmanager decorator for simpler syntax.

In [ ]:
from contextlib import contextmanager

@contextmanager
def database_connection(connection_string):
    """Context manager using decorator."""
    # Setup (before yield)
    conn = psycopg2.connect(connection_string)
    cursor = conn.cursor()
    
    try:
        yield cursor  # This is what 'as var' gets - this is where the code inside the 'with' block runs
        # Success - commit
        conn.commit()
    except Exception:
        # Error - rollback
        conn.rollback()
        raise
    finally:
        # Always cleanup
        cursor.close()
        conn.close()

# Usage - exact same as class version
with database_connection('postgresql://localhost/mydb') as cursor:
    cursor.execute("INSERT INTO users (name) VALUES ('Alice')")

__Structure:__
1. Code before yield = setup (_ _ enter _ _)
2. yield value = what user gets
3. Code after yield = cleanup (_ _ exit _ _)
4. finally block = guranteed cleanup

## Part 6: Practical DE Context Managers

### 1. Timing Block

In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def timer(operation_name: str):
    """Time a block of code."""
    start = time.time()
    try:
        yield
    finally:
        elapsed = time.time() - start
        print(f"{operation_name} took {elapsed:.2f}s")

# Usage
with timer("Data extraction"):
    # Extract data
    time.sleep(2)

with timer("Data transformation"):
    # Transform data
    time.sleep(1)

### 2. Temporary Directory

In [ ]:
import tempfile
import shutil
from pathlib import Path

@contextmanager
def temp_directory():
    """Create temporary directory, auto-delete after use."""
    temp_dir = Path(tempfile.mkdtemp())
    try:
        yield temp_dir
    finally:
        # Delete directory and all contents
        shutil.rmtree(temp_dir)

# Usage
with temp_directory() as tmp:
    # Work with temporary files
    (tmp / 'data.csv').write_text('id,value\n1,100')
    # Process files
# Directory automatically deleted

### 3. File Lock

In [ ]:
import fcntl

@contextmanager
def file_lock(filename: str):
    """Acquire exclusive file lock."""
    f = open(filename, 'a')
    try:
        fcntl.flock(f.fileno(), fcntl.LOCK_EX)
        yield f
    finally:
        fcntl.flock(f.fileno(), fcntl.LOCK_UN)
        f.close()

# Usage - prevents concurrent writes
with file_lock('output.csv') as f:
    f.write('new data\n')
# Lock automatically released

### 4. Database Transaction

In [ ]:
@contextmanager
def transaction(connection):
    """Manage database transaction."""
    cursor = connection.cursor()
    try:
        yield cursor
        connection.commit()
        print("Transaction committed")
    except Exception as e:
        connection.rollback()
        print(f"Transaction rolled back: {e}")
        raise
    finally:
        cursor.close()

# Usage
conn = db.connect('...')
with transaction(conn) as cursor:
    cursor.execute("INSERT INTO orders ...")
    cursor.execute("UPDATE inventory ...")
    # If any query fails, all rolled back

### 5. Supressing Specific Errors

In [ ]:
from contextlib import suppress

# Ignore FileNotFoundError
with suppress(FileNotFoundError):
    os.remove('maybe_exists.txt')
# No error if file doesn't exist

# Multiple exception types
with suppress(KeyError, AttributeError):
    value = data['missing_key'].missing_attr
# Silently continues if error occurs

## Part 7: Multiple Context Managers
You can use multiple context managers in one statement:

In [ ]:
# Open two files
with open('input.csv') as infile, open('output.csv', 'w') as outfile:
    for line in infile:
        outfile.write(line.upper())
# Both files closed

# Database + timing
with database_connection('...') as cursor, timer("Query"):
    cursor.execute("SELECT * FROM huge_table")

## Part 8: Common Patterns

### Part 1: Changing Directory

In [ ]:
import os

@contextmanager
def change_directory(path: str):
    """Temporarily change working directory."""
    original = os.getcwd()
    try:
        os.chdir(path)
        yield
    finally:
        os.chdir(original)

with change_directory('/tmp'):
    # Work in /tmp
    pass
# Back to original directory

### Part 2: Environment Variables

In [ ]:
@contextmanager
def environment_variable(key: str, value: str):
    """Temporarily set environment variable."""
    original = os.environ.get(key)
    os.environ[key] = value
    try:
        yield
    finally:
        if original is None:
            del os.environ[key]
        else:
            os.environ[key] = original

with environment_variable('DB_HOST', 'localhost'):
    # Use test database
    run_tests()
# Original DB_HOST restored

### Pattern 3: Redirecting stdout

In [ ]:
@contextmanager
def redirect_stdout(filename: str):
    """Redirect print statements to file."""
    import sys
    original_stdout = sys.stdout
    with open(filename, 'w') as f:
        sys.stdout = f
        try:
            yield
        finally:
            sys.stdout = original_stdout

with redirect_stdout('output.log'):
    print("This goes to file")
    print("So does this")
# stdout restored